# EVE 310 - Lab 08: Classification thresholds

**Module 3 | 10/15/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab08-classification-thresholds/notebooks/lab08-tutorial.ipynb)

## Learning objectives

By the end of this lab you will be able to:

1. Turn a model's predicted probabilities into yes/no labels with a threshold
2. Count the four outcomes of a classification (true and false positives, true and false negatives) with a loop and an if
3. Compute precision and recall from those counts, and check them against scikit-learn
4. Try many thresholds with a for loop and plot precision and recall against the threshold

## Before you start

1. Click **Copy to Drive** at the top of this window and work in the copy that opens. Colab throws away anything you did not copy when the runtime ends.
2. Run the setup cell below before anything else. It creates `DATA_DIR` and `FIGURES_DIR` for this lab.
3. **This tutorial is typed, not read.** Most code cells are missing one or two lines, marked `--`. In lab, type them along with the instructor, then run the cell. The text above each cell says what the lines do, and **Show the finished cell** below it has the answer if you get stuck or are working on your own. A cell with a `--` still in it cannot run, so **Run all** fails until every blank is filled.
4. Nothing to submit for this notebook. The **activity** notebook is the one you download as `.ipynb` and upload to Gradescope (optional, for feedback).

### Tutorial

Last week's for loop and if statement get their first real job today. A classification model (logistic regression, in lecture) does not answer yes or no. For each well, day or pipe it gives a **probability**, a number between 0 and 1. To act on it you need a **threshold**: above it, call the answer yes (1); otherwise, no (0). The default threshold is 0.5, but nothing says 0.5 is the best choice, and there is no built-in function that tries other thresholds for you. A for loop over a grid of thresholds does it in a few lines, and that is this lab.

The example is the one from the classification lectures: trichloroethylene (TCE), an industrial solvent, in shallow groundwater on Long Island, New York. A well is a **detection** (y = 1) when its TCE concentration is above 3 ppb, and a logistic regression predicts the probability of a detection from the population density near the well. The ten wells below are that model's testing data: the true answer for each well, and the probability the model gave it. If precision and recall are new to you, this notebook defines everything it uses.

## 0. Setup

In [ ]:
# EVE 310 setup - run this cell first, every time you open this notebook.
import pathlib

LAB = "lab08-classification-thresholds"

DATA_DIR = pathlib.Path("data")
FIGURES_DIR = pathlib.Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print(f"Ready. This notebook needs no data files; save any figures to {FIGURES_DIR.resolve()}")

**Type it:** import NumPy as `np` and Matplotlib's pyplot as `plt`, as in every lab since Lab 4.

In [ ]:
# Import NumPy and Matplotlib with their usual aliases
--
--

<details><summary>Show the finished cell</summary>

```python
# Import NumPy and Matplotlib with their usual aliases
import numpy as np
import matplotlib.pyplot as plt
```

</details>

## 1. The data: ten wells, ten probabilities

**Run as is:** `y_true` is the true label of each well (1 = TCE above 3 ppb was measured) and `y_prob` is the model's probability of a detection for that well. `x_density` is the population density the model used, in persons per acre, shown only so you can see the pattern: denser areas, higher probability. The loop prints the three side by side, one well per line.

In [ ]:
# The ten testing wells from lecture
x_density = np.array([2, 5, 11, 13, 6, 3, 16, 5, 18, 11])
y_true = np.array([0, 0, 1, 1, 0, 0, 0, 1, 1, 1])
y_prob = np.array([0.087, 0.158, 0.42, 0.532, 0.191, 0.107, 0.690, 0.159, 0.777, 0.42])

print(len(y_true), 'wells,', y_true.sum(), 'true detections')
print('density  y_true  y_prob')
for i in range(len(y_true)):
    print(f'{x_density[i]:7}  {y_true[i]:6}  {y_prob[i]:6}')

## 2. From a probability to a label: the threshold

A threshold turns probabilities into labels. With a threshold of 0.5, a well is **predicted** to be a detection when its probability is above 0.5. Comparing the whole array with the threshold asks that question for every well at once (Lab 7, section 1) and returns an array of `True` / `False`: `True` means predicted detection (1), `False` means predicted no detection (0). Because `True` counts as 1, `.sum()` on that array is the number of wells flagged.

**Type it:** `y_pred = y_prob > threshold`, and in the last print, `y_pred.sum()`.

In [ ]:
threshold = 0.5

# Predicted label of every well: True (1) above the threshold, False (0) otherwise
--

print('Predicted:', y_pred)
print('True:     ', y_true)
print('Wells flagged as detections:', --)

<details><summary>Show the finished cell</summary>

```python
threshold = 0.5

# Predicted label of every well: True (1) above the threshold, False (0) otherwise
y_pred = y_prob > threshold

print('Predicted:', y_pred)
print('True:     ', y_true)
print('Wells flagged as detections:', y_pred.sum())
```

</details>

Three wells are flagged: the 4th, 7th and 9th. Compare them with the `True` line. The 7th well is flagged but its true label is 0, a false alarm. The 3rd, 8th and 10th wells are true detections that the model left unflagged, misses.

## 3. The four outcomes

Every well lands in one of four boxes, depending on its true label and its predicted label:

| | Predicted 1 (flagged) | Predicted 0 (not flagged) |
| --- | --- | --- |
| **True 1** (detection) | True positive, **TP**: a detection we caught | False negative, **FN**: a detection we missed |
| **True 0** (no detection) | False positive, **FP**: a false alarm | True negative, **TN**: correctly left alone |

Lecture 16 draws the same four boxes as the confusion matrix. Counting them is Lab 7's counting loop with four counters: loop over every well, ask which box it is in with an `if` / `elif` chain, and add 1 to that box. `y_pred[i] == 1` asks whether well `i` was flagged (`True` equals 1 in Python), and `y_true[i] == 1` whether it is a real detection; the two questions are joined with `and`.

**Type it:** the `if` for a true positive (true 1 **and** predicted 1) and the `elif` for a false positive (true 0 and predicted 1). The false negative and true negative branches are written for you.

In [ ]:
tp = 0   # true positives
fp = 0   # false positives
fn = 0   # false negatives
tn = 0   # true negatives

for i in range(len(y_true)):
    --
        tp = tp + 1
    --
        fp = fp + 1
    elif y_true[i] == 1 and y_pred[i] == 0:
        fn = fn + 1
    else:
        tn = tn + 1

print('TP:', tp, ' FP:', fp, ' FN:', fn, ' TN:', tn)
print('The four counts add up to', tp + fp + fn + tn, 'wells')

<details><summary>Show the finished cell</summary>

```python
tp = 0   # true positives
fp = 0   # false positives
fn = 0   # false negatives
tn = 0   # true negatives

for i in range(len(y_true)):
    if y_true[i] == 1 and y_pred[i] == 1:
        tp = tp + 1
    elif y_true[i] == 0 and y_pred[i] == 1:
        fp = fp + 1
    elif y_true[i] == 1 and y_pred[i] == 0:
        fn = fn + 1
    else:
        tn = tn + 1

print('TP:', tp, ' FP:', fp, ' FN:', fn, ' TN:', tn)
print('The four counts add up to', tp + fp + fn + tn, 'wells')
```

</details>

**The array shortcut (run as is):** as in Lab 7, a comparison on the whole array plus `.sum()` counts without a loop. Two conditions on whole arrays are joined with `&` (not `and`), each inside its own parentheses. Use it to check the loop; the activity's check lines do the same.

In [ ]:
print('TP by the array shortcut:', ((y_true == 1) & (y_pred == 1)).sum())
print('FN by the array shortcut:', ((y_true == 1) & (y_pred == 0)).sum())
print('Matches the loop?', tp == ((y_true == 1) & (y_pred == 1)).sum(), fn == ((y_true == 1) & (y_pred == 0)).sum())

## 4. Precision and recall

Two questions summarize the four counts:

- **Precision** = TP / (TP + FP): of the wells we flagged, what fraction really are detections? High precision means few false alarms.
- **Recall** = TP / (TP + FN): of the wells that really are detections, what fraction did we catch? High recall means few misses.

Accuracy, (TP + TN) / all wells, counts right answers of both kinds and is given below. It can mislead when one class is rare: with 1 detection in 100 wells, a model that never flags anything is 99% accurate and useless.

**Type it:** `precision_hand = tp / (tp + fp)` and `recall_hand = tp / (tp + fn)`. Keep the parentheses: `tp / tp + fp` divides first and adds afterwards.

In [ ]:
# Precision: of the flagged wells, the fraction that are true detections
--
# Recall: of the true detections, the fraction that were flagged
--
accuracy_hand = (tp + tn) / len(y_true)

print('Precision:', round(precision_hand, 3))
print('Recall:   ', round(recall_hand, 3))
print('Accuracy: ', round(accuracy_hand, 3))

<details><summary>Show the finished cell</summary>

```python
# Precision: of the flagged wells, the fraction that are true detections
precision_hand = tp / (tp + fp)
# Recall: of the true detections, the fraction that were flagged
recall_hand = tp / (tp + fn)
accuracy_hand = (tp + tn) / len(y_true)

print('Precision:', round(precision_hand, 3))
print('Recall:   ', round(recall_hand, 3))
print('Accuracy: ', round(accuracy_hand, 3))
```

</details>

With a threshold of 0.5: 2 of the 3 flagged wells are real detections (precision 0.667), and only 2 of the 5 real detections were caught (recall 0.4). These are the numbers on the Lecture 16 slide "Selecting the threshold".

## 5. scikit-learn does the same

scikit-learn has `precision_score` and `recall_score`. Both take the true labels first and the predicted labels second, and both accept the `True` / `False` array as it is. One detail: when **no** well is flagged, TP + FP is 0 and precision is 0 / 0. By default sklearn then returns 0 and prints a warning; `zero_division=1` tells it to return 1 instead (no flags, so no false alarms). The sweep in section 6 reaches such thresholds, so this tutorial passes `zero_division=1` every time.

**Type it:** a print of `precision_score(y_true, y_pred, zero_division=1)` and a print of `recall_score(y_true, y_pred)`.

In [ ]:
from sklearn.metrics import precision_score, recall_score

--
--

<details><summary>Show the finished cell</summary>

```python
from sklearn.metrics import precision_score, recall_score

print('Precision:', precision_score(y_true, y_pred, zero_division=1))
print('Recall:   ', recall_score(y_true, y_pred))
```

</details>

## 6. Many thresholds: a for loop

Which threshold is best? Try them all: 0, 0.1, 0.2, ..., 1.0. For each one, predict the labels, then record precision and recall. By hand that means repeating sections 2 to 5 eleven times. A for loop does it with one copy of the code. Three ingredients, all from earlier labs. The new names end in `_array` so that they never collide with `threshold`, the single number from section 2: a loop that compared with `threshold` would quietly use 0.5 on every pass.

1. The grid of thresholds: `np.arange(0, 1.1, 0.1)`. Like `range`, `arange` stops **before** its second number, so the stop is 1.1 to make the grid reach 1.0.
2. Two empty arrays to fill, one slot per threshold: `np.zeros(len(threshold_array))`.
3. A loop over `range(len(threshold_array))` that, on pass `i`, uses `threshold_array[i]` and stores its results in slot `i` of each array: `precision_array[i] = ...`. Filling a slot is new; Lab 7's accumulators grew one number, these arrays keep every pass.

**Type it:** the `threshold_array` grid with `np.arange(0, 1.1, 0.1)`, and the empty `precision_array` with `np.zeros(len(threshold_array))`.

In [ ]:
# The grid of thresholds to try
--
print('Thresholds:', threshold_array)

# Empty arrays, one slot per threshold, for the loop to fill
--
recall_array = np.zeros(len(threshold_array))
print('Empty precision array:', precision_array)

<details><summary>Show the finished cell</summary>

```python
# The grid of thresholds to try
threshold_array = np.arange(0, 1.1, 0.1)
print('Thresholds:', threshold_array)

# Empty arrays, one slot per threshold, for the loop to fill
precision_array = np.zeros(len(threshold_array))
recall_array = np.zeros(len(threshold_array))
print('Empty precision array:', precision_array)
```

</details>

**Your turn:** this one is yours, not the instructor's. On each pass of the loop, in order:

1. predict the labels with the **current** threshold, `threshold_array[i]`, not the whole grid and not `threshold`
2. store the precision in slot `i` of `precision_array`, with `zero_division=1`
3. store the recall in slot `i` of `recall_array`

Replace the three `--`. The lines after the loop print the three arrays side by side, one threshold per line, and check slot 5 (threshold 0.5) against section 4.

In [ ]:
for i in range(len(threshold_array)):
    # 1. labels at this threshold
    y_pred = --
    # 2. precision at this threshold, stored in slot i
    precision_array[i] = --
    # 3. recall at this threshold, stored in slot i
    recall_array[i] = --

print('threshold  precision  recall')
for i in range(len(threshold_array)):
    print(f'{threshold_array[i]:9.1f}  {precision_array[i]:9.3f}  {recall_array[i]:6.3f}')
print('Slot 5 matches section 4?', np.isclose(precision_array[5], precision_hand), np.isclose(recall_array[5], recall_hand))

<details><summary>Show the finished cell</summary>

```python
for i in range(len(threshold_array)):
    # 1. labels at this threshold
    y_pred = y_prob > threshold_array[i]
    # 2. precision at this threshold, stored in slot i
    precision_array[i] = precision_score(y_true, y_pred, zero_division=1)
    # 3. recall at this threshold, stored in slot i
    recall_array[i] = recall_score(y_true, y_pred)

print('threshold  precision  recall')
for i in range(len(threshold_array)):
    print(f'{threshold_array[i]:9.1f}  {precision_array[i]:9.3f}  {recall_array[i]:6.3f}')
print('Slot 5 matches section 4?', np.isclose(precision_array[5], precision_hand), np.isclose(recall_array[5], recall_hand))
```

</details>

## 7. Plot precision and recall against the threshold

**Type it:** two `ax.plot` lines, precision in blue (`color='b'`) and recall in red (`color='r'`), both against `threshold_array`. The labels, title, legend and `savefig` are given, as in Lab 4. The legend names the lines in the order they were plotted, so plot precision first.

In [ ]:
fig, ax = plt.subplots()
--
--
ax.set_xlabel('Threshold')
ax.set_ylabel('Precision, recall')
ax.set_title('Selecting the threshold')
ax.legend(['Precision', 'Recall'])
fig.savefig(FIGURES_DIR / 'lab08_tutorial_threshold.png', dpi=200)

<details><summary>Show the finished cell</summary>

```python
fig, ax = plt.subplots()
ax.plot(threshold_array, precision_array, color='b')
ax.plot(threshold_array, recall_array, color='r')
ax.set_xlabel('Threshold')
ax.set_ylabel('Precision, recall')
ax.set_title('Selecting the threshold')
ax.legend(['Precision', 'Recall'])
fig.savefig(FIGURES_DIR / 'lab08_tutorial_threshold.png', dpi=200)
```

</details>

## 8. Reading the plot

Read it from left to right, as the threshold rises:

- At 0, every well is flagged: recall is 1 (nothing is missed) and precision is 0.5 (half the flags are false alarms).
- Raising the threshold removes flags. **Recall can only fall or stay flat**: a removed flag is either a false alarm gone (recall unchanged) or a detection lost (recall down). **Precision tends to rise**, because the sure-looking wells keep their flags, but it can dip when a real detection loses its flag before a false alarm does, as it does here between 0.4 and 0.6.
- From 0.8 on, no well is flagged: recall is 0, and precision is 1 only by the `zero_division=1` convention.
- From 0.2 to 0.4, precision and recall are both 0.8. For these ten wells a threshold around 0.3 beats the default 0.5.

Which threshold to pick depends on which mistake costs more. A missed detection (FN) leaves people drinking contaminated water; a false alarm (FP) costs one more lab test. Here a miss is worse, so a lower threshold, with high recall, is the safer choice. In the activity the same question comes with a rule: catch at least 80% of the detections, then keep false alarms as few as possible.

**Common mistakes**

| What you see | What went wrong |
| --- | --- |
| `ValueError: operands could not be broadcast together` | `y_prob > threshold_array` inside the loop compares with the whole grid at once. It must be `threshold_array[i]` |
| Every slot of `precision_array` holds the same number | `y_prob > threshold` inside the loop uses `threshold`, the single 0.5 from section 2, on every pass. It must be `threshold_array[i]` |
| `NameError: name 'precision_array' is not defined`, or `IndexError` | The empty arrays were not created before the loop, or have fewer slots than the grid |
| `UndefinedMetricWarning` and a precision of 0 at high thresholds | No well is flagged there, so precision is 0 / 0. Pass `zero_division=1` to `precision_score` |
| A precision above 1 | `tp / tp + fp` divides first and adds afterwards. Keep the parentheses: `tp / (tp + fp)` |
| `ValueError: The truth value of an array ... is ambiguous` | `and` between two whole arrays. On arrays use `&`, with parentheses around each comparison |
| The grid ends at 0.9 | `np.arange(0, 1, 0.1)` stops before 1. Use 1.1 as the stop |
| A count that keeps growing | The `tp = 0` lines sit in another cell (Lab 7's trap). Keep them in the cell with the loop |

## 9. Why this matters

- **Any classifier:** evaluating one means choosing its threshold, and this loop is how it is done. The curve from lecture that plots the true positive rate against the false positive rate (the ROC curve) is the same sweep with two other quantities stored on each pass.
- **Lab 9:** the same loop reads many data files with one copy of the code.
- **Lab 7, again:** `for i in range(len(...))`, `if` / `elif`, counters, and now arrays filled one slot per pass.

Now open the **activity** notebook.

## Wrap-up

Nothing to submit for the tutorial. Keep it open: the activity uses the same loops on all 643 wells of the study.